# 09.03 · Workflow patterns

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] Implement the five core patterns: prompt chaining, routing, parallelization, orchestrator–workers, evaluator–optimizer
- [ ] Choose the **least autonomy that works**: workflows (code decides the path) before agents (model decides)

## Resources
- [Anthropic – Building effective agents (the patterns)](https://www.anthropic.com/engineering/building-effective-agents) — free
- [LangGraph – Workflows and agents](https://langchain-ai.github.io/langgraph/tutorials/workflows/) — free

| Pattern | Use when |
|---|---|
| **Prompt chaining** | task splits into fixed steps; add checks ("gates") between them |
| **Routing** | inputs fall into categories that need different handling (or different models) |
| **Parallelization** | independent subtasks (sectioning) or several attempts to vote on |
| **Orchestrator–workers** | subtasks aren't known in advance; an LLM plans them |
| **Evaluator–optimizer** | clear quality criteria; iterative refinement helps |
| **Agent** | open-ended tasks, unpredictable number of steps |

In [ ]:
import asyncio
import json
from typing import Literal

from openai import AsyncOpenAI
from pydantic import BaseModel

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
ac = AsyncOpenAI(base_url=str(c.base_url), api_key=c.api_key)


def llm(prompt, system=None, max_tokens=600, **kw):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    return c.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens, **EXTRA, **kw).choices[0].message.content


def llm_parse(prompt, schema, max_tokens=800):
    return c.chat.completions.parse(model=MODEL, messages=[{"role": "user", "content": prompt}], response_format=schema,
                                    temperature=0, max_tokens=max_tokens, **EXTRA).choices[0].message.parsed

## 1. Prompt chaining (with a gate)
Outline → check the outline → write → polish. Each step is simpler than the whole; the gate stops bad intermediate output early.

In [ ]:
class Outline(BaseModel):
    title: str
    points: list[str]


topic = "Why every AI app needs an evaluation set"
outline = llm_parse(f"Write a blog post outline (title + 3-5 points) about: {topic}", Outline)
print(outline)

if not 3 <= len(outline.points) <= 5:  # gate: plain code, deterministic
    raise ValueError("outline failed the gate; retry or fix the prompt")

draft = llm(f"Write a ~150-word blog post titled '{outline.title}' covering:\n- " + "\n- ".join(outline.points))
final = llm(f"Edit for clarity and punch. Keep it under 150 words. Return only the post.\n\n{draft}")
print(final)

## 2. Routing
Classify first, then send to a specialised prompt — or a specialised **model** (cheap model for easy queries, strong model for hard ones).

In [ ]:
class Route(BaseModel):
    route: Literal["refund", "technical", "general"]


HANDLERS = {
    "refund": "You handle refunds. Ask for the order number if missing. Never promise a refund before checking eligibility.",
    "technical": "You are a technical support engineer. Ask for error messages and steps to reproduce. Give numbered steps.",
    "general": "You are a friendly support agent. Answer briefly.",
}


def route_and_answer(message):
    r = llm_parse(f"Classify this support message:\n{message}", Route).route
    return r, llm(message, system=HANDLERS[r], max_tokens=200)


for m in ["I want my money back for order 1234", "The app shows error 0x80070005 when syncing", "What are your opening hours?"]:
    r, a = route_and_answer(m)
    print(f"[{r}] {m}\n    {a[:160]!r}\n")

## 3. Parallelization
**Sectioning**: independent parts at once. **Voting**: several attempts, aggregate (like self-consistency, 06.02). Async + gather (01.01).

In [ ]:
async def allm(prompt, max_tokens=300, **kw):
    r = await ac.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=max_tokens, **EXTRA, **kw)
    return r.choices[0].message.content


code = "def login(user, pw):\n    q = f\"SELECT * FROM users WHERE name='{user}' AND pw='{pw}'\"\n    return db.execute(q)"
aspects = ["security vulnerabilities", "performance", "readability and naming"]
reviews = await asyncio.gather(*(allm(f"Review this code ONLY for {a}. 2 bullet points max.\n\n{code}") for a in aspects))
for a, r in zip(aspects, reviews):
    print(f"## {a}\n{r}\n")

votes = await asyncio.gather(*(allm(f"Is this code vulnerable to SQL injection? Answer only YES or NO.\n\n{code}", max_tokens=5, temperature=0.8)
                              for _ in range(5)))
print("votes:", votes)

## 4. Orchestrator–workers
An orchestrator LLM decides the subtasks at runtime; workers execute them (in parallel); a synthesiser combines.

In [ ]:
class Plan(BaseModel):
    subtasks: list[str]


async def orchestrate(task):
    plan = llm_parse(f"Break this task into 2-4 independent research subtasks (one sentence each):\n{task}", Plan)
    print("plan:", plan.subtasks)
    results = await asyncio.gather(*(allm(f"Do this subtask in <=80 words:\n{s}", max_tokens=200) for s in plan.subtasks))
    return llm("Combine these notes into one coherent 120-word answer.\n\n" + "\n\n".join(f"## {s}\n{r}" for s, r in zip(plan.subtasks, results)))


print(await orchestrate("Compare running an LLM locally vs via an API for a small startup."))

## 5. Evaluator–optimizer
A generator writes, an evaluator critiques against explicit criteria, loop until it passes (or N rounds). Works when "good" is checkable.

In [ ]:
class Review(BaseModel):
    passes: bool
    feedback: str


CRITERIA = "under 80 words; mentions the role 'AI engineer'; mentions one concrete project; no clichés like 'passionate' or 'synergy'"


def evaluator_optimizer(task, rounds=3):
    text = llm(task, max_tokens=200)
    for i in range(rounds):
        review = llm_parse(f"Check this text against the criteria: {CRITERIA}.\nCount words carefully.\n\nText:\n{text}", Review)
        words = len(text.split())
        print(f"round {i}: passes={review.passes} words={words} feedback={review.feedback[:120]!r}")
        if review.passes and words < 80:  # trust, but verify the checkable part in code
            return text
        text = llm(f"Rewrite to satisfy: {CRITERIA}.\nFeedback: {review.feedback}\n\nText:\n{text}\n\nReturn only the rewritten text.", max_tokens=200)
    return text


print(evaluator_optimizer("Write a short cover-letter opening for a junior AI engineer job at a fintech."))

Note the hybrid check: the LLM judges fuzzy criteria, **code** checks what code can (word count). LLMs can't count words reliably.

## Exercises
1. Model routing: route "easy" questions to `qwen3:1.7b` and "hard" ones to a hosted model. Measure cost and quality on 20 mixed questions.
2. Rewrite the orchestrator so workers can call `search_docs` (08.06), producing a cited mini-report.
3. Add a max-cost guard to `evaluator_optimizer` and log each round's tokens. How many rounds are typically worth it?

In [ ]:
# your code here